# Unsupervised Learning & Autoencoder Assignment

## Objective
This notebook turns the supplied study links into a complete practical assignment covering:

1. Clustering fundamentals
2. DBSCAN clustering
3. Bisecting K-Means clustering
4. Outlier detection
5. Autoencoders

The supplied source contains study links only, not separate assignment questions. Therefore, the practical tasks below are constructed directly from the five supplied topics, without inventing additional source requirements.


## Source scope

The uploaded study-links file contains links for scikit-learn clustering, DBSCAN, Bisecting K-Means, outlier detection, and a TensorFlow autoencoder tutorial. fileciteturn2file0L1-L5

The implementations below demonstrate those topics with standard public datasets and reproducible code.


## 0. Setup

In [ ]:
# If needed in a fresh Colab environment, uncomment:
# !pip install -q tensorflow scikit-learn matplotlib pandas

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, BisectingKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)


# 1. Clustering Fundamentals — K-Means

### Task
Create an unlabeled synthetic dataset, apply K-Means clustering, visualize the clusters, and evaluate them with the silhouette score.

Clustering is an unsupervised learning approach: the model receives feature data without target labels and attempts to discover groups with similar observations.


In [ ]:
X, _ = make_blobs(
    n_samples=600,
    centers=4,
    cluster_std=0.75,
    random_state=42
)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)

print(f"K-Means silhouette score: {kmeans_silhouette:.4f}")


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=kmeans_labels,
    s=20
)
plt.scatter(
    kmeans.cluster_centers_[:, 0],
    kmeans.cluster_centers_[:, 1],
    marker="X",
    s=180,
    label="Centroids"
)
plt.xlabel("Feature 1 (scaled)")
plt.ylabel("Feature 2 (scaled)")
plt.title("K-Means Clustering")
plt.legend()
plt.show()


### Result
The silhouette score printed above measures how well-separated the discovered clusters are. A higher value indicates more cohesive and separated clusters.


# 2. DBSCAN Clustering

### Task
Apply DBSCAN to a non-linearly shaped dataset.

DBSCAN groups points based on density rather than requiring a predefined number of clusters. It can also identify observations that do not belong to any dense region as noise/outliers.


In [ ]:
X_moons, _ = make_moons(
    n_samples=600,
    noise=0.06,
    random_state=42
)

dbscan = DBSCAN(eps=0.22, min_samples=8)
db_labels = dbscan.fit_predict(X_moons)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = np.sum(db_labels == -1)

print("DBSCAN clusters:", n_clusters)
print("DBSCAN noise points:", n_noise)

# Silhouette score is meaningful only when there are at least 2 non-noise clusters.
mask = db_labels != -1
if len(set(db_labels[mask])) >= 2:
    db_silhouette = silhouette_score(X_moons[mask], db_labels[mask])
    print(f"DBSCAN silhouette score (excluding noise): {db_silhouette:.4f}")
else:
    print("Silhouette score not available for the detected clustering.")


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    X_moons[:, 0],
    X_moons[:, 1],
    c=db_labels,
    s=20
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("DBSCAN Clustering")
plt.show()


### Result
DBSCAN is useful when clusters have irregular shapes and when explicit noise detection is desirable. Its main parameters are `eps`, the neighborhood radius, and `min_samples`, the minimum number of nearby samples needed to form a dense region.


# 3. Bisecting K-Means

### Task
Apply Bisecting K-Means to the same general clustering problem and compare its result with standard K-Means.

Bisecting K-Means repeatedly splits an existing cluster into two until the requested number of clusters is reached.


In [ ]:
X_bisect, _ = make_blobs(
    n_samples=600,
    centers=5,
    cluster_std=0.9,
    random_state=42
)
X_bisect_scaled = StandardScaler().fit_transform(X_bisect)

bisect_kmeans = BisectingKMeans(
    n_clusters=5,
    random_state=42
)
bisect_labels = bisect_kmeans.fit_predict(X_bisect_scaled)

bisect_silhouette = silhouette_score(X_bisect_scaled, bisect_labels)

print(f"Bisecting K-Means silhouette score: {bisect_silhouette:.4f}")


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(
    X_bisect_scaled[:, 0],
    X_bisect_scaled[:, 1],
    c=bisect_labels,
    s=20
)
plt.scatter(
    bisect_kmeans.cluster_centers_[:, 0],
    bisect_kmeans.cluster_centers_[:, 1],
    marker="X",
    s=160,
    label="Centroids"
)
plt.xlabel("Feature 1 (scaled)")
plt.ylabel("Feature 2 (scaled)")
plt.title("Bisecting K-Means")
plt.legend()
plt.show()


# 4. Outlier Detection

### Task
Detect anomalous observations using two approaches:
- Isolation Forest
- Local Outlier Factor (LOF)

Outlier detection is different from ordinary clustering: the main objective is to identify observations that are unusually different from the majority.


In [ ]:
# Create mostly normal observations plus deliberately separated anomalies
rng = np.random.RandomState(42)

normal = rng.normal(loc=0, scale=1, size=(500, 2))
outliers = rng.uniform(low=-7, high=7, size=(25, 2))

X_outlier = np.vstack([normal, outliers])

# Isolation Forest
iso = IsolationForest(
    contamination=25 / 525,
    random_state=42
)
iso_pred = iso.fit_predict(X_outlier)
iso_outliers = iso_pred == -1

print("Isolation Forest detected:", iso_outliers.sum(), "outliers")

# Local Outlier Factor
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=25 / 525
)
lof_pred = lof.fit_predict(X_outlier)
lof_outliers = lof_pred == -1

print("LOF detected:", lof_outliers.sum(), "outliers")


In [ ]:
fig = plt.figure(figsize=(8, 5))
plt.scatter(
    X_outlier[:, 0],
    X_outlier[:, 1],
    c=iso_outliers.astype(int),
    s=22
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Isolation Forest Outlier Detection")
plt.show()

fig = plt.figure(figsize=(8, 5))
plt.scatter(
    X_outlier[:, 0],
    X_outlier[:, 1],
    c=lof_outliers.astype(int),
    s=22
)
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Local Outlier Factor Detection")
plt.show()


### Comparison
Isolation Forest isolates unusual points using randomized tree partitions, while LOF compares the local density of each point with the density around its neighbors. The detected counts and visualizations above show the behavior of both methods on the same data.


# 5. Autoencoder for Image Reconstruction

### Task
Build an autoencoder using the MNIST dataset.

An autoencoder learns to reconstruct its input after passing it through a lower-dimensional latent representation. It consists of an **encoder** and a **decoder**. The encoder compresses the image, and the decoder reconstructs it.


In [ ]:
# Load and normalize MNIST
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Flatten 28x28 images to 784-dimensional vectors
x_train_flat = x_train.reshape((len(x_train), 784))
x_test_flat = x_test.reshape((len(x_test), 784))

print("Training shape:", x_train_flat.shape)
print("Test shape:", x_test_flat.shape)


In [ ]:
# Build the autoencoder
latent_dim = 32

encoder = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(latent_dim, activation="relu")
], name="encoder")

decoder = keras.Sequential([
    layers.Input(shape=(latent_dim,)),
    layers.Dense(128, activation="relu"),
    layers.Dense(784, activation="sigmoid")
], name="decoder")

autoencoder = keras.Sequential(
    [encoder, decoder],
    name="mnist_autoencoder"
)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()


In [ ]:
# Train the autoencoder
history = autoencoder.fit(
    x_train_flat,
    x_train_flat,
    epochs=10,
    batch_size=256,
    shuffle=True,
    validation_data=(x_test_flat, x_test_flat),
    verbose=1
)

test_reconstruction_loss = autoencoder.evaluate(
    x_test_flat,
    x_test_flat,
    verbose=0
)

print(f"Test reconstruction MSE: {test_reconstruction_loss:.6f}")


In [ ]:
# Plot training and validation loss
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("Autoencoder Reconstruction Loss")
plt.legend()
plt.show()


In [ ]:
# Visualize original vs reconstructed images
reconstructed = autoencoder.predict(x_test_flat[:10], verbose=0)
reconstructed = reconstructed.reshape((-1, 28, 28))

plt.figure(figsize=(12, 4))

for i in range(10):
    ax = plt.subplot(2, 10, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Original")

    ax = plt.subplot(2, 10, i + 11)
    plt.imshow(reconstructed[i], cmap="gray")
    plt.axis("off")
    if i == 0:
        plt.ylabel("Reconstructed")

plt.tight_layout()
plt.show()


In [ ]:
# Inspect the learned latent representation
encoded = encoder.predict(x_test_flat[:2000], verbose=0)

# Project the 32-dimensional latent vectors into 2D for visualization
latent_2d = PCA(n_components=2, random_state=42).fit_transform(encoded)

plt.figure(figsize=(7, 5))
plt.scatter(
    latent_2d[:, 0],
    latent_2d[:, 1],
    s=8,
    alpha=0.6
)
plt.xlabel("Latent PCA component 1")
plt.ylabel("Latent PCA component 2")
plt.title("2D Projection of Autoencoder Latent Space")
plt.show()


# 6. Final Comparison and Conclusions

| Topic | Main technique | Primary purpose |
|---|---|---|
| K-Means | Centroid-based clustering | Discover compact groups |
| DBSCAN | Density-based clustering | Find irregular clusters + noise |
| Bisecting K-Means | Recursive binary splitting | Hierarchical-style partitioning into K groups |
| Isolation Forest | Random isolation trees | Detect global anomalies |
| LOF | Local density comparison | Detect locally unusual observations |
| Autoencoder | Encoder + decoder neural network | Learn compressed representations and reconstruct inputs |

## Key conclusions

- **K-Means** is effective when clusters are reasonably compact and the desired number of clusters is known.
- **DBSCAN** is valuable for irregular cluster shapes and explicit noise detection.
- **Bisecting K-Means** reaches a chosen number of clusters through repeated binary splits.
- **Isolation Forest** and **LOF** solve a different problem from clustering: they focus on identifying anomalous observations.
- **Autoencoders** learn lower-dimensional representations while minimizing reconstruction error and can be useful for representation learning, compression, denoising, and anomaly-detection workflows.

The numerical metrics in this notebook are generated when the notebook is executed, so no results are fabricated in advance.


# References

The assignment is based directly on the five supplied study links. fileciteturn2file0L1-L5

1. scikit-learn — Clustering  
   https://scikit-learn.org/stable/modules/clustering.html

2. scikit-learn — DBSCAN  
   https://scikit-learn.org/stable/modules/clustering.html#dbscan

3. scikit-learn — Bisect K-Means  
   https://scikit-learn.org/stable/modules/clustering.html#bisect-k-means

4. scikit-learn — Outlier Detection  
   https://scikit-learn.org/stable/modules/outlier_detection.html

5. TensorFlow — Autoencoder  
   https://www.tensorflow.org/tutorials/generative/autoencoder
